In [ ]:
# ============================================================
# preprocess_castaways.ipynb
# Requires:
#   - /kaggle/input/books/In-search-of-the-castaways.txt
#   - /kaggle/input/name-counts/In_Search_of_the_Castaways_name_counts.csv
# Produces:
#   - /kaggle/working/castaways_constraints.jsonl
# ============================================================

In [ ]:
!pip install -U "transformers>=4.44.0" "accelerate>=0.33.0" --no-deps

In [ ]:
import json
import math
import re
from pathlib import Path
from typing import List, Dict
from tqdm.auto import tqdm
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
BOOK_NAME = "In Search of the Castaways"
BOOK_PATH = Path("/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/In search of the castaways.txt")
NAME_COUNTS_CSV = Path("/kaggle/input/kdsh26-name-counts-csv/In_Search_of_the_Castaways_name_counts.csv")
OUT_PATH = Path("/kaggle/working/castaways_constraints.jsonl")

CHUNK_SIZE = 4000

In [ ]:
LOG_PATH = Path("/kaggle/working/progress_castaways.log")

def log_progress(msg: str):
    """Print + append progress messages."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

In [ ]:
name_df = pd.read_csv(NAME_COUNTS_CSV)

# display(name_df.head(30))  # inspect if needed

# Alias map based on likely characters in Castaways
ALIAS_MAP = {
    "Lord Glenarvan": "Lord Glenarvan",
    "Glenarvan": "Lord Glenarvan",
    "Glenarvan's": "Lord Glenarvan",

    "Captain Grant": "Captain Grant",
    "Grant": "Captain Grant",
    "Robert Grant": "Robert Grant",
    "Robert": "Robert Grant",
    "Mary Grant": "Mary Grant",
    "Mary": "Mary Grant",

    "Lady Helena": "Lady Helena Glenarvan",
    "Lady Helena Glenarvan": "Lady Helena Glenarvan",
    "Helena": "Lady Helena Glenarvan",

    "Jacques Paganel": "Jacques Paganel",
    "Paganel": "Jacques Paganel",

    "John Mangles": "John Mangles",
    "Mangles": "John Mangles",

    "Major MacNabbs": "Major MacNabbs",
    "MacNabb": "Major MacNabbs",
    "MacNabbs": "Major MacNabbs",

    "Wilson": "Wilson",
    "Mulready": "Mulready",
    "Austin": "Austin",

    "Thalcave": "Thalcave",

    "Kai-Koumou": "Kai-Koumou",

    # Multi-name character
    "Tom Ayrton": "Tom Ayrton / Ben Joyce",
    "Tom": "Tom Ayrton / Ben Joyce",
    "Ayrton": "Tom Ayrton / Ben Joyce",
    "Ben Joyce": "Tom Ayrton / Ben Joyce",
    "Ben": "Tom Ayrton / Ben Joyce",
    "Joyce": "Tom Ayrton / Ben Joyce",
}

CANONICAL_CHARS = sorted(set(ALIAS_MAP.values()))


In [ ]:
def load_book(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def chunk_book(text: str, chunk_size: int = CHUNK_SIZE):
    n = math.ceil(len(text) / chunk_size)
    for i in range(n):
        yield i, text[i*chunk_size:(i+1)*chunk_size]

def detect_present_canonicals(chunk_text: str) -> List[str]:
    found = set()
    for surface, canon in ALIAS_MAP.items():
        # disambiguate generic tokens like "Tom" / "Ben" to avoid random matches
        if surface in {"Tom", "Ben"}:
            if not (re.search(r"\bAyrton\b", chunk_text) or re.search(r"\bJoyce\b", chunk_text)):
                continue
        pattern = rf"\b{re.escape(surface)}\b"
        if re.search(pattern, chunk_text):
            found.add(canon)
    return list(found)


In [ ]:
# ----------------------------
# LLM setup (reuse same 8B model)
# ----------------------------

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" #"meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print("Loaded LLM:", MODEL_NAME)

In [ ]:
# ----------------------------
# Prompts for Castaways
# ----------------------------

DIMENSIONS = [
    "birthplace",
    "role",
    "geographic_expertise",
    "tribe_or_nation",
    "languages",
    "moral_arc",
    "criminal_history",
    "core_motivation",
]

SYSTEM_PROMPT = f"""
You are extracting factual constraints about ONE character from
Jules Verne's "In Search of the Castaways".

Focus on dimensions:
- role (noble, geographer, sailor, criminal, guide, Maori chief, etc.)
- geographic_expertise (Pampas, Patagonia, seas, Australia, New Zealand)
- tribe_or_nation (Patagonian, Maori, Scottish, French, etc.)
- languages (Spanish, English, French, indigenous languages)
- moral_arc (e.g., criminal_to_redeemed, loyal_helper)
- criminal_history (for Tom Ayrton / Ben Joyce)
- core_motivation (loyalty, duty, search for Captain Grant)

Output STRICT JSON with schema:

{{
  "character": str,  # canonical name
  "book_name": "In Search of the Castaways",
  "constraints": [
    {{
      "dimension": str,    # one of: {", ".join(DIMENSIONS)},
      "value": str,
      "polarity": "positive" or "negative",
      "evidence_text": str,
      "chapter_id": str
    }}
  ]
}}

Include only facts clearly stated or strongly implied in the excerpt.
"""

USER_PROMPT_TEMPLATE = """
Book: In Search of the Castaways
Canonical character: {character}
Chunk id: {chapter_id}

Excerpt:
\"\"\"{text_chunk}\"\"\"
"""

In [ ]:
# ----------------------------
# LLM helpers
# ----------------------------

def generate_json_response(system_prompt: str, user_prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Explicit attention mask to avoid the warning and ensure correct behavior
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=model.device)

    eos_id = model.config.eos_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos_id

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic; no temperature/top_p needed
            pad_token_id=pad_id,
            eos_token_id=eos_id,
        )

    gen_ids = output_ids[0, input_ids.shape[-1]:]
    out_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return out_text


def extract_first_json(text: str) -> Dict:
    match = re.search(r"\{.*\}", text, flags=re.S)
    candidate = match.group(0) if match else text
    candidate = candidate.strip().strip("`")
    try:
        return json.loads(candidate)
    except Exception:
        return {}

def call_llm_extract_constraints(canonical_char: str, chapter_id: int, text_chunk: str) -> Dict:
    user_prompt = USER_PROMPT_TEMPLATE.format(
        character=canonical_char,
        chapter_id=chapter_id,
        text_chunk=text_chunk,
    )
    raw_text = generate_json_response(SYSTEM_PROMPT, user_prompt, max_new_tokens=512)
    obj = extract_first_json(raw_text)

    if not isinstance(obj, dict):
        obj = {}

    obj.setdefault("character", canonical_char)
    obj.setdefault("book_name", BOOK_NAME)
    obj.setdefault("constraints", [])

    if not isinstance(obj["constraints"], list):
        obj["constraints"] = []

    cleaned_constraints = []
    for c in obj["constraints"]:
        if not isinstance(c, dict):
            continue
        dim  = c.get("dimension")
        val  = c.get("value")
        pol  = c.get("polarity", "positive")
        evid = c.get("evidence_text", "")
        chap = c.get("chapter_id", str(chapter_id))

        if not dim or not val:
            continue

        cleaned_constraints.append({
            "dimension": dim,
            "value": val,
            "polarity": pol if pol in ("positive", "negative") else "positive",
            "evidence_text": evid,
            "chapter_id": str(chap),
        })

    obj["constraints"] = cleaned_constraints
    return obj

In [ ]:
# ----------------------------
# Merge and main
# ----------------------------

def merge_constraints(book_name: str, character: str, objs: List[Dict]) -> Dict:
    merged = []
    seen = set()
    for obj in objs:
        for c in obj.get("constraints", []):
            key = (c["dimension"], c["value"].strip().lower(), c["polarity"])
            if key in seen:
                continue
            seen.add(key)
            merged.append(c)
    return {
        "book_name": book_name,
        "character": character,
        "constraints": merged,
    }

def main():
    text = load_book(BOOK_PATH)
    char_to_objs = {c: [] for c in CANONICAL_CHARS}

    chunks = list(chunk_book(text))  # materialize to know total length for tqdm
    total_chunks = len(chunks)
    log_progress(f"Starting {BOOK_NAME} preprocessing: {total_chunks} chunks")

    for chunk_idx, (chapter_id, chunk) in enumerate(
        tqdm(chunks, desc="Chunks processed", unit="chunk"),
        start=1
    ):
        present = detect_present_canonicals(chunk)
        if not present:
            # log occasionally for empty chunks so you see progress in saved notebook
            if chunk_idx % 25 == 0:
                log_progress(f"Chunk {chunk_idx}/{total_chunks}: no target characters found")
            continue

        log_progress(
            f"Chunk {chunk_idx}/{total_chunks} (id={chapter_id}): "
            f"{len(present)} characters present: {present}"
        )

        # Inner tqdm is optional; you can keep or drop it
        for ci, canon in enumerate(
            tqdm(
                present,
                desc=f"Chars in chunk {chapter_id}",
                unit="char",
                leave=False
            ),
            start=1
        ):
            try:
                obj = call_llm_extract_constraints(canon, chapter_id, chunk)
                char_to_objs[canon].append(obj)
                log_progress(
                    f"  ↳ Processed character {ci}/{len(present)} in chunk {chunk_idx}: {canon}"
                )
            except Exception as e:
                log_progress(f"  ✗ Error for {canon} chunk {chapter_id}: {e}")

    all_outputs = []
    for canon, objs in char_to_objs.items():
        merged = merge_constraints(BOOK_NAME, canon, objs)
        all_outputs.append(merged)
        log_progress(f"Merged constraints for {canon}: {len(merged['constraints'])} constraints")

    with OUT_PATH.open("w", encoding="utf-8") as f:
        for obj in all_outputs:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    log_progress(f"Saved constraints to {OUT_PATH}")



In [ ]:
if __name__ == \"__main__\":
    main()